# RQ1 Experimental Workflow

## Research Question

**How does active Gaussian payload affect steady-state rendering speed on a fixed GPU?**

## Workflow

1. **Fix the GPU**
   - Select one GPU for all experiments.
   - Record the GPU model, VRAM, CUDA version, driver version, and power mode.

   ↓

2. **Fix the renderer settings**
   - Keep the image resolution, depth-sample count, field of view, camera pose, cube dimensions, tile size, Mahalanobis cutoff, and activation functions constant.

   ↓

3. **Select one Gaussian scene**
   - Use one representative Gaussian block or camera-centred region.
   - Use the same scene and camera pose for every run.

   ↓

4. **Create controlled Gaussian subsets**
   - Generate deterministic subsets of the full Gaussian population.
   - Example retention ratios:
     - 10%
     - 20%
     - 30%
     - 40%
     - 50%
     - 60%
     - 75%
     - 100%

   ↓

5. **Calculate active count and payload**
   - Record the active Gaussian count.
   - Calculate the active payload:

     \[
     B_{\text{active}}
     =
     N_{\text{active}}
     \times
     B_{\text{Gaussian}}
     \]

   - Report the payload in bytes and MiB.

   ↓

6. **Prepare the renderer**
   - Upload the selected Gaussian subset to the GPU.
   - Allocate the required CUDA buffers.
   - Build tile or spatial preprocessing structures.
   - Exclude this preparation stage from steady-state timing.

   ↓

7. **Warm up the renderer**
   - Render 20 untimed frames.
   - Discard all warm-up measurements.

   ↓

8. **Measure per-frame CUDA time**
   - Render 300 measured frames.
   - Measure each frame separately using CUDA events.
   - Exclude loading, preprocessing, allocation, and file-writing time.

   ↓

9. **Repeat five times**
   - Repeat each payload configuration five times.
   - Keep the Gaussian subset and all renderer settings unchanged.

   ↓

10. **Aggregate FPS statistics**
    - Calculate:
      - Mean FPS
      - Median FPS
      - FPS standard deviation
      - p5 FPS
      - p95 FPS
      - Mean render time
      - Median render time
      - p95 render time

   ↓

11. **Plot payload versus FPS**
    - Plot active Gaussian count against median FPS.
    - Plot active payload in MiB against median FPS.
    - Add error bars using variation across repeated runs.

   ↓

12. **Fit the scalability model**
    - Fit a linear model:

      \[
      T_{\text{render}}
      =
      \beta_0
      +
      \beta_1N_{\text{active}}
      +
      \epsilon
      \]

    - Optionally compare it with a quadratic model.
    - Report \(R^2\), adjusted \(R^2\), RMSE, and AIC.

   ↓

13. **Answer RQ1**
    - State how FPS changes as the Gaussian payload increases.
    - Determine whether render time grows linearly or nonlinearly.
    - Report the observed rate of performance degradation.
    - Do not discuss GPU-memory normalisation, cross-GPU transferability, or the 30-FPS deployment threshold.

```mermaid
flowchart TD
    A[Fix GPU]
    B[Fix renderer settings]
    C[Select one Gaussian scene]
    D[Create controlled Gaussian subsets]
    E[Calculate active count and MiB]
    F[Prepare renderer]
    G[Warm up]
    H[Measure per-frame CUDA time]
    I[Repeat five times]
    J[Aggregate FPS statistics]
    K[Plot payload versus FPS]
    L[Fit scalability model]
    M[Answer RQ1]

    A --> B
    B --> C
    C --> D
    D --> E
    E --> F
    F --> G
    G --> H
    H --> I
    I --> J
    J --> K
    K --> L
    L --> M

In [1]:
#write the code to Record the GPU model, VRAM, CUDA version, driver version, and power mode.

from sympy import python

import subprocess
def get_gpu_info():
    try:
        # Get GPU model and VRAM
        gpu_info = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']).decode('utf-8').strip().split('\n')
        gpu_model, vram = gpu_info[0].split(', ')
        
        # Get CUDA version
        cuda_version = subprocess.check_output(['nvcc', '--version']).decode('utf-8').strip().split('\n')[-1].split()[-1]
        
        # Get driver version
        driver_version = subprocess.check_output(['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader']).decode('utf-8').strip()
        
        # Get power mode
        power_mode = subprocess.check_output(['nvidia-smi', '--query-gpu=power.limit', '--format=csv,noheader']).decode('utf-8').strip()
        
        return {
            'GPU Model': gpu_model,
            'VRAM': vram,
            'CUDA Version': cuda_version,
            'Driver Version': driver_version,
            'Power Mode': power_mode
        }
    except Exception as e:
        return {'error': str(e)}

if __name__ == "__main__":
    gpu_info = get_gpu_info()
    for key, value in gpu_info.items():
        print(f"{key}: {value}")

GPU Model: NVIDIA RTX 5000 Ada Generation
VRAM: 32760 MiB
CUDA Version: cuda_12.9.r12.9/compiler.36037853_0
Driver Version: 580.126.09
Power Mode: 250.00 W


In [ ]:
# Keep the image resolution, depth-sample count, field of view, camera pose, cube dimensions, tile size, Mahalanobis cutoff, and activation functions constant. based on 